# 2026 enrollment predictions

Reload the saved Poisson evaluation models for pre-acceptance and post-acceptance, and the gradient boosted model with acceptance and deposit features for post-deposit. Sum expected group counts or enrollment probabilities, then round the annual total. Results are displayed only in this notebook; no prediction files are written.

The Poisson models were trained on 2022–2025. Prefer notebook 04's chronological validation artifact when available; the older gradient boosted artifact was fitted using a random split and does not provide an independent 2026 evaluation.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits

BASE_DIR = Path.cwd()
PREDICTION_YEAR = 2026
poisson_bundles = {
    checkpoint: joblib.load(BASE_DIR / "results" / "total_enrollment" / f"{checkpoint}_evaluation_model.joblib")
    for checkpoint in ["pre_acceptance", "post_acceptance"]
}
gradient_path = BASE_DIR / "results" / "with_acceptance_fee_and_deposits_validation_model.joblib"
if not gradient_path.exists():
    gradient_path = BASE_DIR / "results" / "with_acceptance_fee_and_deposits_model.joblib"
gradient_bundle = joblib.load(gradient_path)


In [2]:
def aggregate_for_poisson(applicants, bundle):
    """Reproduce notebook 05's aggregation using the saved model recipe."""
    work = applicants.copy()
    recipe = bundle["aggregation"]
    aggregations = {"applicants": ("student_id", "size")}
    aggregations.update({
        f"{column}_mean": (column, "mean")
        for column in recipe["mean_columns"]
    })
    aggregations.update({
        f"{column}_share": (column, "mean")
        for column in recipe["share_columns"]
    })
    for column in ["hs_gpa", "act_cmpst"]:
        name = f"{column}_missing_share"
        work[name] = work[column].isna().astype(float)
        aggregations[name] = (name, "mean")
    work["early_application_share"] = (
        work["app_yr"].lt(work["entry_year"]).astype(float)
        .mask(work["app_yr"].isna())
    )
    aggregations["early_application_share"] = ("early_application_share", "mean")
    if bundle["checkpoint"] == "post_acceptance":
        aggregations["admitted_share"] = ("admitted", "mean")
        for label in ["paid", "deferred", "waived"]:
            name = f"acceptance_{label}_share"
            work[name] = work["acceptance_fee"].eq(bundle["fee_codes"][label]).astype(float)
            aggregations[name] = (name, "mean")
        work["acceptance_recorded_share"] = work["acceptance_fee"].ne(-1).astype(float)
        aggregations["acceptance_recorded_share"] = ("acceptance_recorded_share", "mean")
    groups = (
        work.groupby(bundle["group_keys"], dropna=False, observed=True)
        .agg(**aggregations).reset_index()
        .sort_values(bundle["group_keys"]).reset_index(drop=True)
    )
    assert groups["applicants"].sum() == len(applicants)
    return groups


In [3]:
dataset = pd.read_excel(BASE_DIR / "cleaned_dataset.xlsx")
applicants_2026 = dataset.loc[dataset["entry_year"].eq(PREDICTION_YEAR)].drop(
    columns=["enrolled"], errors="ignore"
).copy()
if applicants_2026.empty:
    raise ValueError("No applicants found for entry year 2026.")
if applicants_2026["student_id"].isna().any() or applicants_2026["student_id"].duplicated().any():
    raise ValueError("Applicant IDs must be present and unique.")

predictions = []
for checkpoint, bundle in poisson_bundles.items():
    if max(bundle["training_years"]) >= PREDICTION_YEAR:
        raise ValueError("Use a Poisson evaluation model trained before 2026.")
    groups = aggregate_for_poisson(applicants_2026, bundle)
    with threadpool_limits(limits=2):
        rates = bundle["model"].predict(groups)
    counts = np.clip(rates * groups["applicants"].to_numpy(), 0, groups["applicants"].to_numpy())
    predictions.append({
        "checkpoint": checkpoint,
        "model": "Poisson regression",
        "entry_year": PREDICTION_YEAR,
        "applicants": len(applicants_2026),
        "predicted_enrollment": float(counts.sum()),
    })

# Match notebook 04's feature derivation, order, and missing-category handling.
features = gradient_bundle["selected_features"]
gradient_frame = applicants_2026.copy()
if "visit_yrs_before_entry" in features:
    gradient_frame["visit_yrs_before_entry"] = gradient_frame["entry_year"] - gradient_frame["visit_yr"]
values = gradient_frame.loc[:, features].apply(pd.to_numeric, errors="raise").astype(float)
values = values.replace([np.inf, -np.inf], np.nan)
for column in {"app_fee", "acceptance_fee", "residency", "sex", "college"}.intersection(features):
    values[column] = values[column].mask(values[column].eq(-1))
with threadpool_limits(limits=2):
    model = gradient_bundle["model"]
    enrolled_class = list(model.classes_).index(1)
    probabilities = model.predict_proba(values)[:, enrolled_class]
predictions.append({
    "checkpoint": "post_deposit",
    "model": "Gradient boosted classifier",
    "entry_year": PREDICTION_YEAR,
    "applicants": len(applicants_2026),
    "predicted_enrollment": float(probabilities.sum()),
})

prediction_results = pd.DataFrame(predictions)
assert np.isfinite(prediction_results["predicted_enrollment"]).all()
prediction_results["rounded_enrollment"] = np.rint(
    prediction_results["predicted_enrollment"]
).astype(int)
display(prediction_results.round({"predicted_enrollment": 2}))
if "training_entry_years" not in gradient_bundle:
    print("Post-deposit uses the older saved gradient boosted model; its random training split may include 2026 applicants.")

,checkpoint,model,entry_year,applicants,predicted_enrollment,rounded_enrollment
0,pre_acceptance,Poisson regression,2026,13967,4600.10,4600
1,post_acceptance,Poisson regression,2026,13967,4461.43,4461
2,post_deposit,Gradient boosted classifier,2026,13967,4358.07,4358


Post-deposit uses the older saved gradient boosted model; its random training split may include 2026 applicants.
